In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

STOCKS_CSV = "nlp_clean_stock_symbols.csv"

def load_stocks(path=STOCKS_CSV):
    df = pd.read_csv(path)

    # Choose a text field for retrieval
    df["kw_text"] = df["document_clean_tfidf"].fillna(df["document_raw"]).fillna("")
    df["sem_text"] = df["document_raw"].fillna(df["document_clean_tfidf"]).fillna("")

    # Keep only usable rows
    df = df[(df["kw_text"].str.len() > 0) & (df["sem_text"].str.len() > 0)].reset_index(drop=True)
    return df

def build_hybrid_index(df,
                       sbert_model="sentence-transformers/all-MiniLM-L6-v2",
                       tfidf_min_df=2,
                       tfidf_ngram_range=(1,2)):
    # TF-IDF for keyword similarity
    vectorizer = TfidfVectorizer(min_df=tfidf_min_df, max_df=0.95, ngram_range=tfidf_ngram_range)
    X_tfidf = vectorizer.fit_transform(df["kw_text"].astype(str))

    # SBERT embeddings for semantic similarity 
    model = SentenceTransformer(sbert_model)
    X_emb = model.encode(
        df["sem_text"].astype(str).tolist(),
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True
    ).astype(np.float32)

    return vectorizer, X_tfidf, model, X_emb

df_stocks = load_stocks(STOCKS_CSV)
vectorizer, X_tfidf, sbert, X_emb = build_hybrid_index(df_stocks)

'''
#persist to disk for production use
import joblib
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
joblib.dump(df_stocks, "stocks_df.joblib")
np.save("stocks_sbert_embeddings.npy", X_emb)
'''

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 81/81 [00:50<00:00,  1.61it/s]


'\n#persist to disk for production use\nimport joblib\njoblib.dump(vectorizer, "tfidf_vectorizer.joblib")\njoblib.dump(df_stocks, "stocks_df.joblib")\nnp.save("stocks_sbert_embeddings.npy", X_emb)\n'

model

In [2]:
import numpy as np

def _minmax(x):
    x = np.asarray(x, dtype=np.float32)
    mn, mx = x.min(), x.max()
    return (x - mn) / (mx - mn + 1e-9)

def build_query_from_df_text_inputs_row(row):
    # Use text_input if present; otherwise combine topics
    t = str(row.get("text_input", "")).strip()
    if t:
        return t
    p = str(row.get("primary_topic", "")).strip()
    s = str(row.get("secondary_topic", "")).strip()
    return (p + " " + s).strip()

def hybrid_retrieve(query, df_stocks, vectorizer, X_tfidf, sbert, X_emb,
                    top_k=50,
                    w_sem=0.65, w_kw=0.35,
                    mmr_lambda=0.80,
                    candidate_pool=500,
                    sector_cap=10):
    """
    w_sem/w_kw: weight semantic vs keyword similarity
    mmr_lambda: higher => more relevance, lower => more diversity
    candidate_pool: retrieve this many by relevance before diversifying to top_k
    sector_cap: max results per sector (set None to disable)
    """

    # keyword similarity (TF-IDF cosine for L2-normalized TF-IDF is dot product)
    q_tfidf = vectorizer.transform([query])
    kw_scores = (X_tfidf @ q_tfidf.T).toarray().ravel()

    # semantic similarity (cosine via normalized vectors)
    q_emb = sbert.encode([query], normalize_embeddings=True).astype(np.float32)  # (1, d)
    sem_scores = (X_emb @ q_emb.T).ravel()

    # combine (normalize each so neither dominates)
    kw_n = _minmax(kw_scores)
    sem_n = _minmax(sem_scores)
    scores = w_sem * sem_n + w_kw * kw_n

    # candidate set
    pool = min(candidate_pool, len(df_stocks))
    cand_idx = np.argpartition(-scores, pool-1)[:pool]
    cand_idx = cand_idx[np.argsort(-scores[cand_idx])]

    # MMR diversification using embedding similarity to reduce redundancy
    selected = []
    selected_set = set()

    # Precompute candidate embeddings
    cand_emb = X_emb[cand_idx]

    # Sector tracking
    sector_counts = {}
    def sector_ok(i):
        if sector_cap is None:
            return True
        sec = df_stocks.loc[i, "sector"]
        sec = sec if isinstance(sec, str) and sec else "Unknown"
        return sector_counts.get(sec, 0) < sector_cap

    for _ in range(min(top_k, len(cand_idx))):
        best_i = None
        best_mmr = -1e9

        for pos, i in enumerate(cand_idx):
            if i in selected_set:
                continue
            if not sector_ok(i):
                continue

            rel = scores[i]

            if not selected:
                mmr = rel
            else:
                # max similarity to already selected (redundancy)
                sim_to_selected = np.max(X_emb[i] @ X_emb[selected].T)
                mmr = mmr_lambda * rel - (1 - mmr_lambda) * sim_to_selected

            if mmr > best_mmr:
                best_mmr = mmr
                best_i = i

        if best_i is None:
            break

        selected.append(best_i)
        selected_set.add(best_i)
        sec = df_stocks.loc[best_i, "sector"]
        sec = sec if isinstance(sec, str) and sec else "Unknown"
        sector_counts[sec] = sector_counts.get(sec, 0) + 1

    out = df_stocks.loc[selected, ["symbol","company_name","sector","industry"]].copy()
    out["hybrid_score"] = scores[selected]
    out = out.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
    return out

test

In [3]:
import pandas as pd
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer

# Load what saved (or reuse in-memory objects)
df_stocks = joblib.load("stocks_df.joblib")
vectorizer = joblib.load("tfidf_vectorizer.joblib")
X_emb = np.load("stocks_sbert_embeddings.npy").astype(np.float32)
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Rebuild TF-IDF matrix (or persist it similarly via scipy sparse save)
from sklearn.feature_extraction.text import TfidfVectorizer
X_tfidf = vectorizer.transform(df_stocks["kw_text"].astype(str))

df_queries = pd.read_csv("df_text_inputs.csv")

for i in range(5):
    q = build_query_from_df_text_inputs_row(df_queries.iloc[i])
    print("\nQUERY:", q)
    res = hybrid_retrieve(q, df_stocks, vectorizer, X_tfidf, sbert, X_emb,
                          top_k=50, w_sem=0.65, w_kw=0.35,
                          mmr_lambda=0.80, sector_cap=10)
    print(res.head(10))


QUERY: I'm interested in companies focused on music streaming.
  symbol                       company_name                  sector  \
0    TME  Tencent Music Entertainment Group  Communication Services   
1  TCMEF  Tencent Music Entertainment Group  Communication Services   
2  KUKEY             Kuke Music Holding Ltd  Communication Services   
3    WMG           Warner Music Group Corp.  Communication Services   
4   RSVR              Reservoir Media, Inc.  Communication Services   
5  RSVRW              Reservoir Media, Inc.                     NaN   
6    LVO                      LiveOne, Inc.  Communication Services   
7   SONG               Music Licensing Inc.  Communication Services   
8   DSNY     DESTINY MEDIA TECHNOLOGIES INC              Technology   
9  MMTRS                  MILLS MUSIC TRUST             Industrials   

                         industry  hybrid_score  
0  Internet Content & Information      0.997045  
1  Internet Content & Information      0.997045  
2   